# Narcolepsy Predictive Model

## Longitudinal feature extraction

In [ ]:
import polars as pl
import pandas as pd
import sys
from pathlib import Path

data_path = Path('data/predictive-modeling') # path to data directory

# please extract features using NarcolepsyModel for all the EHR data for the patients in the cohort
# example code to do this is provided in 'discriminative-modeling/discriminative-example.ipynb', final cell
# the code here uses the features extracted from that notebook

In [ ]:
features = pl.read_parquet(data_path / 'features.parquet')
annot = pl.read_parquet(data_path / 'diagnosis_annotation.parquet')
groupings = pl.read_parquet(data_path / 'groupings.parquet')

## create predictive features for NT1 +, -, controls

In [19]:
annot.filter(
    pl.col('diagnosis') == 'NT1'
)

diagnosis_note_id,diagnosis,bdsp_patient_id,id,date
str,str,i64,str,date
"""Notes_1129886449_17631067619_2…","""NT1""",150027448,"""150027448""",2011-10-13
"""Notes_1129906307_1570443541_20…","""NT1""",150047446,"""150047446""",2011-05-22
"""Notes_1129901044_1879859080_20…","""NT1""",150042036,"""150042036""",2011-11-20
"""Notes_1129910829_2088444741_20…","""NT1""",150051725,"""150051725""",2011-12-09
"""Notes_1129916704_2090319785_20…","""NT1""",150057720,"""150057720""",2011-08-15
…,…,…,…,…
"""I0006_179000970_2563652167_201…","""NT1""",179000970,"""179000970""",2012-09-08
"""I0006_179017315_14070018069_20…","""NT1""",179017315,"""179017315""",2020-09-29
"""I0006_179010680_3981809715_201…","""NT1""",179010680,"""179010680""",2014-04-30


In [ ]:
# features for NT1+ patients
features.join(
    annot.filter(
        pl.col('diagnosis') == 'NT1'
    ).select(['id','date']),
    on='id',
    how='right',
).with_columns(
    pl.when(pl.col('date') < pl.col('date_right')).then(0).otherwise(1).alias('n+_state'), # labels visits as 0 if they occur before the diagnosis date, and 1 if they occur on or after the diagnosis date
).drop('date_right').group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt1/n+_features_3.parquet'
)

# features for NT1- patients, who never receive the diagnosis
features.join(
    groupings.filter(
        pl.col('group') == 'NT1-'
    ).drop('group'),
    on='id',
    how='right',
).with_columns(
    pl.lit(0).alias('n+_state'), # all visits for NT1- patients are labeled as 0, since they never receive the diagnosis
).group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt1/n-_features_3.parquet'
)

# features for controls
features.join(
    groupings.filter(
        pl.col('group') == 'NT1_controls'
    ).drop('group'),
    on='id',
    how='right',
).with_columns(
    pl.lit(0).alias('n+_state'),
).group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt1/controls_features_3.parquet'
)

## create predictive features for NT2IH +, -, controls

In [31]:
annot.filter(
    pl.col('diagnosis') == 'NT2IH'
)

diagnosis_note_id,diagnosis,bdsp_patient_id,id,date
str,str,i64,str,date
"""Notes_1129912911_11770874232_2…","""NT2IH""",150054127,"""150054127""",2011-04-22
"""Notes_1129864804_1872747006_20…","""NT2IH""",150005951,"""150005951""",2011-04-06
"""Notes_1130843091_2222247218_20…","""NT2IH""",150984197,"""150984197""",2011-06-13
"""Notes_1129909454_2083299982_20…","""NT2IH""",150050968,"""150050968""",2011-11-03
"""Notes_1129882263_2435198613_20…","""NT2IH""",150023600,"""150023600""",2011-04-02
…,…,…,…,…
"""I0006_179000770_4941555715_201…","""NT2IH""",179000770,"""179000770""",2014-04-26
"""I0006_179002483_5951587526_201…","""NT2IH""",179002483,"""179002483""",2014-09-20
"""I0006_179013592_4754871681_201…","""NT2IH""",179013592,"""179013592""",2014-03-01


In [ ]:
# features for NT2IH+ patients
features.join(
    annot.filter(
        pl.col('diagnosis') == 'NT2IH'
    ).select(['id','date']),
    on='id',
    how='right',
).with_columns(
    pl.when(pl.col('date') < pl.col('date_right')).then(0).otherwise(1).alias('n+_state'),
).drop('date_right').group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt2ih/n+_features_3.parquet'
)

# features for NT2IH- patients, who never receive the diagnosis
features.join(
    groupings.filter(
        pl.col('group') == 'NT2IH-'
    ).drop('group'),
    on='id',
    how='right',
).with_columns(
    pl.lit(0).alias('n+_state'), # all visits for NT2IH- patients are labeled as 0, since they never receive the diagnosis
).group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt2ih/n-_features_3.parquet'
)

# features for controls
features.join(
    groupings.filter(
        pl.col('group') == 'NT2IH_controls'
    ).drop('group'),
    on='id',
    how='right',
).with_columns(
    pl.lit(0).alias('n+_state'),
).group_by(
    ['id', 'date']
).agg(
    pl.all().exclude(['id', 'date']).max(), # combines all features extracted from notes in a single day 
).sort(['id', 'date']).group_by('id', maintain_order=True).agg(
    'date',
    'n+_state',
    (pl.col('date') - pl.col('date').min()).dt.total_days().alias('days_since_first_visit'),
    pl.row_index().alias('num_visits_since_first_visit'),
    pl.all().exclude(['id', 'date', 'n+_state']).cum_sum() # cumulative sum of features across visits, to capture history
).explode(pl.all().exclude('id')).write_parquet(
    data_path / 'nt2ih/controls_features_3.parquet'
)